# Kerbeus - Derm7pt Multimodal Pipeline

This notebook implements a multimodal skin lesion classification pipeline on the **Derm7pt** dataset, combining dermoscopy images and clinical tabular metadata.

## Structure
| Cell | Contents |
|------|----------|
| 1 | Imports & Global Config |
| 2 | Tabular Preprocessor & Data Loading |
| 3 | Dataset, Transforms & Sampler |
| 4 | Fragility-Aware Sampler |
| 5 | Loss, Metrics & Reporting Helpers |
| 6 | Image Backbone (Dual InceptionV3) |
| 7 | FT-Transformer Tabular Encoder |
| 8 | Cross-Modal Attention Fusion |
| 9 | Triple CLIP Head |
| 10 | Baseline & Kerbeus Model Definitions |
| 11 | Gradient & Modality Contribution Utilities |
| 12 | Training Loops (Baseline & Kerbeus) |
| 13 | Evaluation & Main Pipeline |
| 14 | Curriculum Config & Perturbation Applicators |
| 15 | Perturbed Dataset & Collation |
| 16 | Reliability Head & Gated Model |
| 17 | Curriculum Training |
| 18 | Inference Battery & Final Evaluation |
| 19 | Per-Sample Reliability Gate Demo |

## Step 1 — Imports & Global Config

Load all required libraries and define the central `CFG` dataclass that controls every hyperparameter used throughout the notebook.

In [1]:
import os, warnings, time, math
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as T
from torchvision.models import inception_v3, Inception_V3_Weights

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False

class CFG:
    BASE      = Path("/kaggle/input/datasets/menakamohanakumar/derm7pt/release_v0")
    IMG_DIR   = BASE / "images"
    META_CSV  = BASE / "meta/meta.csv"
    TRAIN_IDX = BASE / "meta/train_indexes.csv"
    VALID_IDX = BASE / "meta/valid_indexes.csv"
    TEST_IDX  = BASE / "meta/test_indexes.csv"

    IMG_SIZE  = 299
    MEAN      = [0.485, 0.456, 0.406]
    STD       = [0.229, 0.224, 0.225]

    CAT_COLS  = [
        "vascular_structures", "blue_whitish_veil", "pigment_network",
        "management", "streaks", "dots_and_globules", "elevation",
        "regression_structures", "pigmentation",
        "level_of_diagnostic_difficulty", "location",
    ]
    NUM_COLS  = ["seven_point_score"]

    EMB_DIM       = 16        
    FT_HIDDEN     = 128       
    FT_HEADS      = 4         
    FT_LAYERS     = 3         
    FT_DROPOUT    = 0.15      
    TAB_HIDDEN    = [256, 128, 64]   
    TAB_DROPOUT   = 0.2

    D_MODEL       = 256       
    ATTN_HEADS    = 8         
    ATTN_DROPOUT  = 0.10      

    EPOCHS         = 50
    EARLY_STOP_PAT = 7
    PHASE1_END     = 5
    PHASE2_END     = 30

    BASELINE_EPOCHS     = 50
    BASELINE_EARLY_STOP = 7

    BATCH_SIZE   = 8
    LR_IMAGE     = 1e-4
    LR_TAB       = 1e-3
    LR_FUSION    = 3e-4
    LR_ATTENTION = 5e-5   
    WEIGHT_DECAY = 1e-4

    NUM_WORKERS  = 2
    SEED         = 7

    R_DIM       = 512    
    CLIP_DIM    = 512    
    LAMBDA_CLIP = 0.05   
    FRAG_ALPHA  = 0.3
    DROP_PROB   = 0.15

    DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CKPT_DIR = Path("/kaggle/working/checkpoints")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

## Step 2 — Tabular Preprocessor & Data Loading

Fit label encoders and a standard scaler on training tabular features, then load and split the Derm7pt metadata CSV into train / val / test frames. Diagnosis labels are merged into five coarse classes: MEL, NEV, BCC, SK, MISC.

In [2]:
class TabularPreprocessor:
    def __init__(self):
        self.label_encoders: dict[str, LabelEncoder] = {}
        self.scaler        = StandardScaler()
        self.fill_values   : dict[str, object]       = {}
        self.cat_dims      : list[int]               = []
        self.tab_dim       : int                     = 0

    def fit_transform(self, df: pd.DataFrame):
        cat_parts, num_parts = [], []
        self.cat_dims = []
        for col in CFG.CAT_COLS:
            series   = df[col].astype(str)
            mode_val = series.mode()[0]
            self.fill_values[col] = mode_val
            series   = series.fillna(mode_val)
            le       = LabelEncoder()
            codes    = le.fit_transform(series)
            self.label_encoders[col] = le
            self.cat_dims.append(len(le.classes_))
            cat_parts.append(codes.reshape(-1, 1))
        for col in CFG.NUM_COLS:
            vals = df[col].fillna(df[col].median()).values.reshape(-1, 1)
            self.fill_values[col] = df[col].median()
            num_parts.append(vals)
        cat_arr = np.hstack(cat_parts).astype(np.int64)
        num_arr = self.scaler.fit_transform(
                      np.hstack(num_parts).astype(np.float32))
        self.tab_dim = len(CFG.CAT_COLS) * CFG.EMB_DIM + len(CFG.NUM_COLS)
        return cat_arr, num_arr

    def transform(self, df: pd.DataFrame):
        cat_parts, num_parts = [], []
        for col in CFG.CAT_COLS:
            series = df[col].astype(str).fillna(str(self.fill_values[col]))
            le     = self.label_encoders[col]
            codes  = series.map(
                lambda x, le=le: le.transform([x])[0]
                                 if x in le.classes_ else 0
            ).values.astype(np.int64)
            cat_parts.append(codes.reshape(-1, 1))
        for col in CFG.NUM_COLS:
            vals = df[col].fillna(
                       self.fill_values[col]).values.reshape(-1, 1)
            num_parts.append(vals)
        cat_arr = np.hstack(cat_parts).astype(np.int64)
        num_arr = self.scaler.transform(
                      np.hstack(num_parts).astype(np.float32))
        return cat_arr, num_arr


def prepare_data():
    print("▶  Loading metadata …")
    meta      = pd.read_csv(CFG.META_CSV)
    train_idx = pd.read_csv(CFG.TRAIN_IDX).values.flatten()
    valid_idx = pd.read_csv(CFG.VALID_IDX).values.flatten()
    test_idx  = pd.read_csv(CFG.TEST_IDX).values.flatten()

    train_df = meta.iloc[train_idx].copy()
    valid_df = meta.iloc[valid_idx].copy()
    test_df  = meta.iloc[test_idx].copy()

    def merge_diagnosis(df):
        df = df.copy()
        melanoma = [
            "melanoma", "melanoma (0.76 to 1.5 mm)", "melanoma (in situ)",
            "melanoma (less than 0.76 mm)", "melanoma (more than 1.5 mm)",
            "melanoma metastasis",
        ]
        nevus = [
            "clark nevus", "combined nevus", "congenital nevus",
            "dermal nevus", "recurrent nevus", "reed or spitz nevus",
            "blue nevus",
        ]
        misc = [
            "dermatofibroma", "lentigo", "melanosis",
            "miscellaneous", "vascular lesion",
        ]
        df["diagnosis"] = df["diagnosis"].replace(melanoma, "MEL")
        df["diagnosis"] = df["diagnosis"].replace(nevus,    "NEV")
        df["diagnosis"] = df["diagnosis"].replace(misc,     "MISC")
        df["diagnosis"] = df["diagnosis"].replace(
            {"basal cell carcinoma": "BCC", "seborrheic keratosis": "SK"})
        return df

    train_df = merge_diagnosis(train_df)
    valid_df = merge_diagnosis(valid_df)
    test_df  = merge_diagnosis(test_df)

    le_target = LabelEncoder()
    y_train   = le_target.fit_transform(train_df["diagnosis"])
    valid_df  = valid_df[valid_df["diagnosis"].isin(le_target.classes_)].copy()
    test_df   = test_df[test_df["diagnosis"].isin(le_target.classes_)].copy()
    y_valid   = le_target.transform(valid_df["diagnosis"])
    y_test    = le_target.transform(test_df["diagnosis"])
    print(f"   Classes ({len(le_target.classes_)}): {list(le_target.classes_)}")

    tab_prep               = TabularPreprocessor()
    X_tr_cat, X_tr_num     = tab_prep.fit_transform(train_df)
    X_val_cat, X_val_num   = tab_prep.transform(valid_df)
    X_test_cat, X_test_num = tab_prep.transform(test_df)

    print(f"   Train: {X_tr_cat.shape[0]} | "
          f"Val: {X_val_cat.shape[0]} | Test: {X_test_cat.shape[0]}")
    print(f"   Cat dims: {tab_prep.cat_dims}")

    return (train_df, valid_df, test_df,
            X_tr_cat, X_tr_num, y_train,
            X_val_cat, X_val_num, y_valid,
            X_test_cat, X_test_num, y_test,
            le_target, tab_prep)

## Cell 3 — Dataset, Transforms & Class Sampler

Define the PyTorch `Dataset` that jointly loads dual dermoscopy images plus tabular features, the image augmentation pipelines for train vs eval modes, and the inverse-frequency class sampler used during baseline training.

In [3]:
def get_transforms(mode: str) -> T.Compose:
    if mode == "train":
        return T.Compose([
            T.RandomResizedCrop(
                CFG.IMG_SIZE, scale=(0.7, 1.0),
                interpolation=T.InterpolationMode.BILINEAR),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=20),
            T.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.2, hue=0.05),
            T.ToTensor(),
            T.Normalize(mean=CFG.MEAN, std=CFG.STD),
        ])
    return T.Compose([
        T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE),
                 interpolation=T.InterpolationMode.BILINEAR),
        T.ToTensor(),
        T.Normalize(mean=CFG.MEAN, std=CFG.STD),
    ])


class Derm7ptDataset(Dataset):
    def __init__(self, df, cat_features, num_features, labels, transform):
        self.df           = df.reset_index(drop=True)
        self.cat_features = cat_features
        self.num_features = num_features
        self.labels       = labels
        self.transform    = transform

    def __len__(self): return len(self.df)

    def _load_img(self, rel_path):
        return self.transform(Image.open(CFG.IMG_DIR / rel_path).convert("RGB"))

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        derm_t   = self._load_img(row["derm"])
        clinic_t = self._load_img(row["clinic"])
        cat_t    = torch.tensor(self.cat_features[idx], dtype=torch.long)
        num_t    = torch.tensor(self.num_features[idx], dtype=torch.float32)
        label_t  = torch.tensor(self.labels[idx],       dtype=torch.long)
        return derm_t, clinic_t, cat_t, num_t, label_t


def make_class_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts  = np.bincount(labels)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float64),
        num_samples = len(sample_weights),
        replacement = True,
    )

## Step 4 — Fragility-Aware Sampler

Compute a per-sample *fragility* score, i.e., the gap between the top predicted probability and the true class probability and use it to up weight hard samples in the training sampler. This is recomputed every five epochs during Phase 2.

In [4]:
@torch.no_grad()
def compute_fragility_sampler(model, dataset: Dataset,
                               y_train: np.ndarray,
                               device, alpha: float = CFG.FRAG_ALPHA
                               ) -> WeightedRandomSampler:
    """
    Fragility = gap between max-class prob and true-class prob.
    ONLY used to reweight the sampler — never in the loss.
    """
    model.eval()
    loader = DataLoader(
        dataset,
        batch_size  = CFG.BATCH_SIZE * 2,
        shuffle     = False,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = True,
    )
    all_fragility: list[float] = []

    for derm, clinic, tab_cat, tab_num, labels in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        labels       = labels.to(device)

        with autocast():
            outputs      = model(derm, clinic, tab_cat, tab_num)
            final_logits = outputs[4]

        probs = F.softmax(final_logits, dim=1)
        pos   = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        neg   = probs.max(dim=1).values
        frag  = (neg - pos).cpu().numpy()
        all_fragility.extend(frag.tolist())

    fragility          = np.array(all_fragility)
    class_counts       = np.bincount(y_train)
    class_weights      = 1.0 / np.maximum(class_counts, 1)
    class_w_per_sample = class_weights[y_train]
    frag_w             = 1.0 + alpha * fragility
    final_w            = class_w_per_sample * frag_w

    return WeightedRandomSampler(
        weights     = torch.tensor(final_w, dtype=torch.float64),
        num_samples = len(final_w),
        replacement = True,
    )

## Step 5 — Loss Function, Metrics & Reporting Helpers

Define the `AsymmetricLoss` (separate focusing for positive vs negative classes), the standard multi-class metric bundle, and the table printing utilities.

In [5]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=2.0, gamma_pos=1.0, eps=0.0):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.eps = gamma_neg, gamma_pos, eps

    def forward(self, logits, targets):
        B, C = logits.shape
        y    = F.one_hot(targets, num_classes=C).float()
        p    = F.softmax(logits, dim=1)
        p_m  = torch.clamp(p - self.eps, min=0)
        p_pos = torch.where(y == 1, p,   torch.zeros_like(p))
        p_neg = torch.where(y == 0, p_m, torch.zeros_like(p_m))
        loss_pos = (y       * torch.log(torch.clamp(p_pos,   min=1e-8))
                            * (1 - p_pos) ** self.gamma_pos)
        loss_neg = ((1 - y) * torch.log(torch.clamp(1-p_neg, min=1e-8))
                            * p_neg       ** self.gamma_neg)
        return -(loss_pos + loss_neg).mean()


def compute_metrics(y_true, y_pred, class_names=None, verbose=False):
    f1_per_class = f1_score(y_true, y_pred, average=None,       zero_division=0)
    macro_f1     = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    weighted_f1  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    metrics = {
        "accuracy"    : accuracy_score(y_true, y_pred),
        "precision"   : precision_score(y_true, y_pred,
                                        average="macro", zero_division=0),
        "recall"      : recall_score(y_true, y_pred,
                                     average="macro", zero_division=0),
        "f1"          : macro_f1,
        "weighted_f1" : weighted_f1,
        "f1_per_class": f1_per_class,
    }
    if verbose:
        print("\n  ── Per-Class F1 ──────────────────────────────────────────")
        for i, f1 in enumerate(f1_per_class):
            name = class_names[i] if class_names else f"Class {i}"
            bar  = "█" * int(f1 * 20)
            print(f"    {name:20s}  F1 = {f1:.4f}  {bar}")
        print(f"    {'─'*40}")
        print(f"    {'Macro  F1':20s}  {macro_f1:.4f}")
        print(f"    {'Weighted F1':20s}  {weighted_f1:.4f}\n")
    return metrics


def print_metrics(split_name, metrics):
    print(f"  ► {split_name}")
    print(f"    Accuracy    : {metrics['accuracy']:.4f}")
    print(f"    Precision   : {metrics['precision']:.4f}")
    print(f"    Recall      : {metrics['recall']:.4f}")
    print(f"    Macro-F1    : {metrics['f1']:.4f}")
    print(f"    Weighted-F1 : {metrics['weighted_f1']:.4f}\n")

## Step 6 — Image Backbone (Dual InceptionV3)

Load two pretrained InceptionV3 backbones, one for dermoscopy (`derm`) images and one for clinical (`clinic`) images. Their global average pooled features are concatenated and projected to `R_DIM` via `combine_visual`, producing `img_feat` used by downstream fusion.

In [6]:
class InceptionBase(nn.Module):
    def __init__(self):
        super().__init__()
        base = inception_v3(weights=Inception_V3_Weights.DEFAULT)
        base.aux_logits = False
        self.features = nn.Sequential(
            base.Conv2d_1a_3x3, base.Conv2d_2a_3x3, base.Conv2d_2b_3x3,
            nn.MaxPool2d(3, stride=2),
            base.Conv2d_3b_1x1, base.Conv2d_4a_3x3,
            nn.MaxPool2d(3, stride=2),
            base.Mixed_5b, base.Mixed_5c, base.Mixed_5d,
            base.Mixed_6a, base.Mixed_6b, base.Mixed_6c,
            base.Mixed_6d, base.Mixed_6e,
            base.Mixed_7a, base.Mixed_7b, base.Mixed_7c,
        )

    def forward(self, x): return self.features(x)


class DiagnosisMultimodalNet(nn.Module):
    """Dual InceptionV3 image branch. Returns same 6-tuple."""
    def __init__(self, num_classes: int):
        super().__init__()
        f_dim = 2048
        self.backbone_d     = InceptionBase()
        self.backbone_c     = InceptionBase()
        self.L_d_conv       = nn.Conv2d(f_dim, num_classes, kernel_size=1)
        self.L_c_conv       = nn.Conv2d(f_dim, num_classes, kernel_size=1)
        self.bn_d           = nn.BatchNorm1d(f_dim)
        self.bn_c           = nn.BatchNorm1d(f_dim)
        self.combine_visual = nn.Sequential(
            nn.Linear(f_dim * 2, CFG.R_DIM),
            nn.ReLU(),
            nn.BatchNorm1d(CFG.R_DIM),
        )
        self.L_dc_linear = nn.Linear(CFG.R_DIM, num_classes)

    def forward(self, x_d, x_c):
        feat_d = self.backbone_d(x_d)
        feat_c = self.backbone_c(x_c)
        out_d  = F.adaptive_avg_pool2d(
                     self.L_d_conv(feat_d), (1, 1)).view(x_d.size(0), -1)
        out_c  = F.adaptive_avg_pool2d(
                     self.L_c_conv(feat_c), (1, 1)).view(x_c.size(0), -1)
        gap_d  = F.adaptive_avg_pool2d(feat_d, (1, 1)).view(x_d.size(0), -1)
        gap_c  = F.adaptive_avg_pool2d(feat_c, (1, 1)).view(x_c.size(0), -1)
        img_feat = self.combine_visual(
            torch.cat([self.bn_d(gap_d), self.bn_c(gap_c)], dim=1))
        out_dc   = self.L_dc_linear(img_feat)
        return out_d, out_c, out_dc, gap_d, gap_c, img_feat

## Step 7 — FT-Transformer Tabular Encoder

Replaces the flat TabularEmbedding + MLP. Each feature (categorical or numeric) is tokenised to `EMB_DIM`, then `FT_LAYERS` transformer encoder layers run inter-feature self-attention. Mean-pooling over the token dimension produces `tab_feat` with depth comparable to the pretrained CNN.

In [7]:
class FTTransformerEncoder(nn.Module):

    def __init__(self, cat_dims: list[int], num_classes: int):
        super().__init__()
        n_cat = len(cat_dims)
        n_num = len(CFG.NUM_COLS)

        self.cat_embeds = nn.ModuleList([
            nn.Embedding(dim + 1, CFG.EMB_DIM) for dim in cat_dims
        ])
        self.num_projectors = nn.ModuleList([
            nn.Linear(1, CFG.EMB_DIM) for _ in range(n_num)
        ])

        n_tokens = n_cat + n_num    

        self.token_proj = nn.Linear(CFG.EMB_DIM, CFG.FT_HIDDEN)

        self.pos_bias = nn.Parameter(torch.zeros(1, n_tokens, CFG.FT_HIDDEN))
        nn.init.trunc_normal_(self.pos_bias, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = CFG.FT_HIDDEN,
            nhead           = CFG.FT_HEADS,
            dim_feedforward = CFG.FT_HIDDEN * 4,
            dropout         = CFG.FT_DROPOUT,
            activation      = "gelu",
            batch_first     = True,   
            norm_first      = True,   
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=CFG.FT_LAYERS)

        self.norm       = nn.LayerNorm(CFG.FT_HIDDEN)
        self.classifier = nn.Sequential(
            nn.Linear(CFG.FT_HIDDEN, CFG.FT_HIDDEN // 2),
            nn.GELU(),
            nn.Dropout(CFG.FT_DROPOUT),
            nn.Linear(CFG.FT_HIDDEN // 2, num_classes),
        )

        self.out_dim = CFG.FT_HIDDEN

    def forward(self, x_cat: torch.Tensor,
                x_num: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        tokens = []

        for i, emb in enumerate(self.cat_embeds):
            tokens.append(emb(x_cat[:, i]))          

        for i, proj in enumerate(self.num_projectors):
            tokens.append(proj(x_num[:, i:i+1]))     

        x = torch.stack(tokens, dim=1)               
        x = self.token_proj(x)                       
        x = x + self.pos_bias                        

        x = self.transformer(x)                      
        tab_feat = self.norm(x.mean(dim=1))          
        logits   = self.classifier(tab_feat)         

        return logits, tab_feat

## Step 8 — Cross-Modal Attention Fusion

Replaces the simple concat + linear FusionHead. Both modalities are projected to `D_MODEL` and stacked into a 2-token sequence, then multi-head self-attention lets each modality attend to the other before the residual + LayerNorm. The concatenated output feeds the final classification MLP.

In [8]:
class CrossModalAttentionFusion(nn.Module):

    def __init__(self, img_dim: int, tab_dim: int, num_classes: int):
        super().__init__()

        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, CFG.D_MODEL),
            nn.LayerNorm(CFG.D_MODEL),
        )
        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, CFG.D_MODEL),
            nn.LayerNorm(CFG.D_MODEL),
        )

        self.attn = nn.MultiheadAttention(
            embed_dim   = CFG.D_MODEL,
            num_heads   = CFG.ATTN_HEADS,
            dropout     = CFG.ATTN_DROPOUT,
            batch_first = True,
        )

        self.norm    = nn.LayerNorm(CFG.D_MODEL)
        self.dropout = nn.Dropout(CFG.ATTN_DROPOUT)

        fused_dim = CFG.D_MODEL * 2
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

        self.out_dim = fused_dim

    def forward(self, img_feat: torch.Tensor,
                tab_feat: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B = img_feat.size(0)

        img_tok = self.img_proj(img_feat).unsqueeze(1)   
        tab_tok = self.tab_proj(tab_feat).unsqueeze(1)   
        tokens  = torch.cat([img_tok, tab_tok], dim=1)   

        attn_out, _ = self.attn(tokens, tokens, tokens)  
        attn_out    = self.dropout(attn_out)

        tokens = self.norm(tokens + attn_out)             

        fused_feat = tokens.view(B, -1)                   
        logits     = self.classifier(fused_feat)
        return logits, fused_feat

## Step 9 — Triple CLIP Head

Three independent projectors map `img_feat`, `tab_feat`, and `fused_feat` into a shared L2-normalised `CLIP_DIM` space. The loss is the mean of three pairwise cosine distances, creating a symmetric alignment triangle that prevents any branch from drifting.

In [9]:
class TripleCLIPHead(nn.Module):
    
    def __init__(self, img_dim: int, tab_dim: int, fused_dim: int):
        super().__init__()

        def _make_proj(in_dim: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Linear(in_dim, 512),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(512, CFG.CLIP_DIM),
            )

        self.proj_img   = _make_proj(img_dim)
        self.proj_tab   = _make_proj(tab_dim)
        self.proj_fused = _make_proj(fused_dim)

    def forward(self, img_feat: torch.Tensor,
                tab_feat: torch.Tensor,
                fused_feat: torch.Tensor) -> torch.Tensor:
        z_img   = F.normalize(self.proj_img(img_feat),     dim=-1)
        z_tab   = F.normalize(self.proj_tab(tab_feat),     dim=-1)
        z_fused = F.normalize(self.proj_fused(fused_feat), dim=-1)

        loss_it = 1.0 - (z_img   * z_tab  ).sum(dim=1).mean()
        loss_if = 1.0 - (z_img   * z_fused).sum(dim=1).mean()
        loss_tf = 1.0 - (z_tab   * z_fused).sum(dim=1).mean()

        return (loss_it + loss_if + loss_tf) / 3.0

## Step 10 — Model Definitions: Baseline & Kerbeus

`BaselineFusionModel` is a simple scalar-weighted average of image and tabular logits, used as the comparison anchor.`RepairedFusionModel (Kerbeus)` wires together all upgrade modules and adds per-sample modality dropout on branch logits (never on raw features) plus optional NaN debug assertions.

In [10]:
class BaselineFusionModel(nn.Module):
    def __init__(self, tab_dim: int, num_classes: int):
        super().__init__()
        self.net = DiagnosisMultimodalNet(num_classes)
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes),
        )
        self.w_img = nn.Parameter(torch.tensor(0.5))
        self.w_tab = nn.Parameter(torch.tensor(0.5))

    def forward(self, derm, clinic, tab_cat, tab_num):
        tab = torch.cat([tab_cat.float(), tab_num], dim=1)
        out_d, out_c, out_dc, _, _, _ = self.net(derm, clinic)
        out_tab      = self.mlp(tab)
        final_logits = self.w_img * out_dc + self.w_tab * out_tab
        return out_d, out_c, out_dc, out_tab, final_logits

In [11]:
class RepairedFusionModel(nn.Module):

    def __init__(self, cat_dims: list[int], num_classes: int):
        super().__init__()
        self.num_classes = num_classes
        self._debug      = bool(int(os.environ.get("TORCH_DEBUG", "0")))

        self.net = DiagnosisMultimodalNet(num_classes)

        self.ft_encoder = FTTransformerEncoder(cat_dims, num_classes)
        tab_dim         = self.ft_encoder.out_dim   

        self.fusion = CrossModalAttentionFusion(
            img_dim     = CFG.R_DIM,
            tab_dim     = tab_dim,
            num_classes = num_classes,
        )
        fused_dim = self.fusion.out_dim   

        self.clip_head = TripleCLIPHead(
            img_dim   = CFG.R_DIM,
            tab_dim   = tab_dim,
            fused_dim = fused_dim,
        )

    def _assert_finite(self, t: torch.Tensor, name: str):
        if self._debug and not torch.isfinite(t).all():
            raise RuntimeError(
                f"[NaN/Inf] '{name}'  min={t.min():.4f}  max={t.max():.4f}")

    def forward(self, derm, clinic, tab_cat, tab_num):
        B = derm.size(0)

        out_d, out_c, out_dc_raw, _, _, img_feat = self.net(derm, clinic)

        out_tab_raw, tab_feat = self.ft_encoder(tab_cat, tab_num)

        self._assert_finite(out_dc_raw,  "out_dc_raw")
        self._assert_finite(out_tab_raw, "out_tab_raw")

        if self.training:
            p        = CFG.DROP_PROB
            r        = torch.rand(B, device=derm.device)
            drop_img = r < p
            drop_tab = (r >= p) & (r < 2 * p)
        else:
            drop_img = torch.zeros(B, dtype=torch.bool, device=derm.device)
            drop_tab = torch.zeros(B, dtype=torch.bool, device=derm.device)

        out_dc  = out_dc_raw.clone();  out_dc[drop_img]  = 0.0
        out_tab = out_tab_raw.clone(); out_tab[drop_tab] = 0.0

        final_logits, fused_feat = self.fusion(img_feat, tab_feat)
        self._assert_finite(final_logits, "final_logits")

        clip_loss = self.clip_head(img_feat, tab_feat, fused_feat)
        self._assert_finite(clip_loss, "clip_loss")

        return out_d, out_c, out_dc, out_tab, final_logits, clip_loss


def set_backbones_trainable(model, trainable: bool):
    for name, module in model.named_modules():
        if name.endswith(("backbone_d", "backbone_c")):
            for p in module.parameters():
                p.requires_grad = trainable

## Step 11 — Gradient Norm & Modality Contribution Utilities

Two diagnostic tools: `compute_branch_grad_norms` computes L2 gradient norms for the image vs tabular branch after each backward pass to detect dominance; `compute_logit_contributions` measures each branch's share of the absolute logit magnitude at evaluation time.

In [12]:
def compute_branch_grad_norms(model) -> dict[str, float]:
    def _norm(params):
        total = sum(
            p.grad.detach().norm(2).item() ** 2
            for p in params
            if p.grad is not None and torch.isfinite(p.grad).all()
        )
        return total ** 0.5

    grad_img = _norm(model.net.parameters()) if hasattr(model, "net") else 0.0
    tab_mod  = model.ft_encoder if hasattr(model, "ft_encoder") else \
               model.mlp        if hasattr(model, "mlp")        else None
    grad_tab = _norm(tab_mod.parameters()) if tab_mod else 0.0
    grad_tab = max(grad_tab, 1e-6)
    return {
        "grad_img"  : grad_img,
        "grad_tab"  : grad_tab,
        "grad_ratio": grad_img / grad_tab,
    }


@torch.no_grad()
def compute_logit_contributions(model, loader, device) -> dict[str, float]:
    model.eval()
    img_pcts, tab_pcts = [], []
    for derm, clinic, tab_cat, tab_num, _ in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        with autocast():
            outputs = model(derm, clinic, tab_cat, tab_num)
        out_dc  = outputs[2]
        out_tab = outputs[3]
        norm_img = out_dc.abs().mean(dim=1)
        norm_tab = out_tab.abs().mean(dim=1)
        denom    = norm_img + norm_tab + 1e-8
        img_pcts.append((norm_img / denom).mean().item())
        tab_pcts.append((norm_tab / denom).mean().item())
    img_pct = float(np.mean(img_pcts)) * 100
    tab_pct = float(np.mean(tab_pcts)) * 100
    return {"img_pct": img_pct, "tab_pct": tab_pct}


def print_modality_balance(contrib: dict[str, float]):
    img_pct = contrib["img_pct"]
    tab_pct = contrib["tab_pct"]
    img_flag = " OK" if 55 <= img_pct <= 70 else " ERR"
    tab_flag = " OK" if 30 <= tab_pct <= 45 else " ERR"
    print(f"  ── Modality Contribution ────────────────────────────────────")
    print(f"    Image    contribution : {img_pct:.2f}%  (healthy: 55–70%){img_flag}")
    print(f"    Tabular  contribution : {tab_pct:.2f}%  (healthy: 30–45%){tab_flag}")

## Step 12 — Training Loops (Baseline & Kerbeus)

Three training functions: the vanilla baseline loop (single flat loss sum), the loop with two-sided adaptive gradient balancing (scaling down whichever branch dominates), and a shared `evaluate` function used by both. The v5 loop applies `w_img` / `w_tab` multipliers symmetrically when `grad_ratio` falls outside the [1/3, 3] corridor.

In [13]:
def train_one_epoch_baseline(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss    = 0.0
    grad_img_list = []
    grad_tab_list = []

    for derm, clinic, tab_cat, tab_num, labels in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        labels       = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out_d, out_c, out_dc, out_tab, out_final = model(
                derm, clinic, tab_cat, tab_num)
            loss = (criterion(out_d,     labels) +
                    criterion(out_c,     labels) +
                    criterion(out_dc,    labels) +
                    criterion(out_tab,   labels) +
                    criterion(out_final, labels))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gn = compute_branch_grad_norms(model)
        grad_img_list.append(gn["grad_img"])
        grad_tab_list.append(gn["grad_tab"])
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_gi = float(np.mean(grad_img_list))
    avg_gt = float(np.mean(grad_tab_list))
    return total_loss / len(loader), {
        "grad_img"  : avg_gi,
        "grad_tab"  : avg_gt,
        "grad_ratio": avg_gi / max(avg_gt, 1e-6),
    }

def train_one_epoch_repaired(model, loader, optimizer, criterion,
                                  scaler, device):
    model.train()
    total_loss    = 0.0
    grad_img_list = []
    grad_tab_list = []
    prev_ratio    = 1.0

    for derm, clinic, tab_cat, tab_num, labels in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        labels       = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out_d, out_c, out_dc, out_tab, final_logits, clip_loss = model(
                derm, clinic, tab_cat, tab_num)

            loss_img    = (criterion(out_d,  labels) +
                           criterion(out_c,  labels) +
                           criterion(out_dc, labels))
            loss_tab    = criterion(out_tab,      labels)
            loss_fusion = criterion(final_logits, labels)

            w_img = min(1.0, 3.0 / prev_ratio) if prev_ratio > 3.0   else 1.0
            w_tab = min(1.0, prev_ratio / 3.0)  if prev_ratio < (1/3) else 1.0

            loss = (w_img             * loss_img
                    + w_tab           * loss_tab
                    +                   loss_fusion
                    + CFG.LAMBDA_CLIP * clip_loss)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        gn         = compute_branch_grad_norms(model)
        prev_ratio = gn["grad_ratio"]
        grad_img_list.append(gn["grad_img"])
        grad_tab_list.append(gn["grad_tab"])

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_gi = float(np.mean(grad_img_list))
    avg_gt = float(np.mean(grad_tab_list))
    return total_loss / len(loader), {
        "grad_img"  : avg_gi,
        "grad_tab"  : avg_gt,
        "grad_ratio": avg_gi / max(avg_gt, 1e-6),
    }

@torch.no_grad()
def evaluate(model, loader, criterion, device, class_names=None, verbose=False):
    model.eval()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for derm, clinic, tab_cat, tab_num, labels in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        labels       = labels.to(device)

        with autocast():
            outputs = model(derm, clinic, tab_cat, tab_num)

        out_d, out_c, out_dc, out_tab, out_final = outputs[:5]

        with autocast():
            loss = (criterion(out_d,     labels) +
                    criterion(out_c,     labels) +
                    criterion(out_dc,    labels) +
                    criterion(out_tab,   labels) +
                    criterion(out_final, labels))

        total_loss += loss.item()
        all_preds.extend(out_final.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    y_true  = np.array(all_labels)
    y_pred  = np.array(all_preds)
    metrics = compute_metrics(y_true, y_pred,
                              class_names=class_names, verbose=verbose)
    return total_loss / len(loader), metrics

## Step 13 — Full Training Pipelines & Main Entrypoint

Wire everything together. `train_model_baseline` runs a flat cosine-LR baseline. `train_model_repaired` runs the 3-phase schedule (backbone frozen → unfrozen + fragility sampling → LR annealing). The `main()` function orchestrates both pipelines, runs modality ablation on the best repaired model, and calls `print_final_report`.

In [14]:
def train_model_baseline(model, train_loader, val_loader, test_loader,
                          le_target=None, device=CFG.DEVICE):
    model     = model.to(device)
    criterion = AsymmetricLoss(gamma_neg=2.0, gamma_pos=1.0, eps=0.0)
    optimizer = AdamW(model.parameters(),
                      lr=CFG.LR_IMAGE, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=CFG.BASELINE_EPOCHS, eta_min=1e-6)
    scaler    = GradScaler()

    best_f1       = 0.0
    patience_ctr  = 0
    ckpt_path     = CFG.CKPT_DIR / "baseline_fusion_best.pt"
    epoch_grad_history = []

    print(f"\n{'='*70}")
    print(f"  Training ► VANILLA BASELINE  (no CLIP · no fragility · flat LR)")
    print(f"{'='*70}")
    print(f"  {'Ep':>4}  {'TrLoss':>8}  {'ValLoss':>8}  {'F1':>7}  "
          f"{'GradImg':>9}  {'GradTab':>9}  {'Ratio':>7}")
    print(f"  {'─'*62}")

    for epoch in range(1, CFG.BASELINE_EPOCHS + 1):
        t0 = time.time()
        tr_loss, grad_stats   = train_one_epoch_baseline(
            model, train_loader, optimizer, criterion, scaler, device)
        val_loss, val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        epoch_grad_history.append(grad_stats)

        if val_metrics["f1"] > best_f1:
            best_f1      = val_metrics["f1"]
            patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
            flag = " ✓"
        else:
            patience_ctr += 1
            flag = f" [{patience_ctr}/{CFG.BASELINE_EARLY_STOP}]"

        elapsed = time.time() - t0
        print(f"  {epoch:4d}  {tr_loss:8.4f}  {val_loss:8.4f}  "
              f"{val_metrics['f1']:7.4f}  "
              f"{grad_stats['grad_img']:9.4f}  "
              f"{grad_stats['grad_tab']:9.4f}  "
              f"{grad_stats['grad_ratio']:7.2f}  "
              f"[{elapsed:.0f}s]{flag}")

        if patience_ctr >= CFG.BASELINE_EARLY_STOP:
            print(f"\n  [Early Stop] Stopping at epoch {epoch}.")
            break

    print(f"\n  Loading best checkpoint (Val F1 = {best_f1:.4f}) …")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    class_names = list(le_target.classes_) if le_target else None
    print(f"\n  ── Test Set — Per-Class F1 (Baseline) ──────────────────────")
    _, test_metrics = evaluate(model, test_loader, criterion, device,
                                class_names=class_names, verbose=True)
    print_metrics("Test (Baseline)", test_metrics)

    contrib  = compute_logit_contributions(model, val_loader, device)
    print_modality_balance(contrib)

    avg_grad = {
        "grad_img"  : float(np.mean([g["grad_img"]   for g in epoch_grad_history])),
        "grad_tab"  : float(np.mean([g["grad_tab"]   for g in epoch_grad_history])),
        "grad_ratio": float(np.mean([g["grad_ratio"] for g in epoch_grad_history])),
    }
    return test_metrics, avg_grad, contrib

In [15]:
def train_model_repaired(model, base_train_loader, val_loader, test_loader,
                              ds_train=None, y_train=None,
                              le_target=None, device=CFG.DEVICE):
    model     = model.to(device)
    criterion = AsymmetricLoss(gamma_neg=2.0, gamma_pos=1.0, eps=0.0)

    optimizer = AdamW([
        {"params": model.net.parameters(),
         "lr": CFG.LR_IMAGE},
        {"params": model.ft_encoder.parameters(),
         "lr": CFG.LR_ATTENTION},   # FT-Transformer: conservative LR
        {"params": model.fusion.parameters(),
         "lr": CFG.LR_ATTENTION},   # cross-modal attention: conservative LR
        {"params": model.clip_head.parameters(),
         "lr": CFG.LR_FUSION},
    ], weight_decay=CFG.WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)
    scaler    = GradScaler()

    best_f1            = 0.0
    patience_ctr       = 0
    ckpt_path          = CFG.CKPT_DIR / "repaired_fusion_best.pt"
    epoch_grad_history = []
    current_loader     = base_train_loader
    phase3_lr_applied  = False

    kw_loader = dict(batch_size=CFG.BATCH_SIZE,
                     num_workers=CFG.NUM_WORKERS,
                     pin_memory=True)

    print(f"\n{'='*70}")
    print(f"  Training ► REPAIRED")
    print(f"  (FT-Transformer · CrossModalAttn · TripleCLIP · "
          f"2-sided balance · 3-phase)")
    print(f"{'='*70}")
    print(f"  {'Ep':>4}  {'TrLoss':>8}  {'ValLoss':>8}  {'F1':>7}  "
          f"{'GradImg':>9}  {'GradTab':>9}  {'Ratio':>7}  {'Phase':>7}")
    print(f"  {'─'*68}")

    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()

        if epoch == 1:
            set_backbones_trainable(model, False)
            phase_tag = "Phase 1"
            print("  [Phase 1] Backbone frozen — training tab + fusion + CLIP")
        elif epoch == CFG.PHASE1_END + 1:
            set_backbones_trainable(model, True)
            phase_tag = "Phase 2"
            print("  [Phase 2] Backbone unfrozen — fragility sampling active")
        elif epoch > CFG.PHASE2_END and not phase3_lr_applied:
            for pg in optimizer.param_groups:
                pg["lr"] = max(pg["lr"] * 0.1, 1e-7)
            phase3_lr_applied = True
            phase_tag = "Phase 3"
            print("  [Phase 3] LR × 0.1 — stabilising gradients")
        elif epoch <= CFG.PHASE1_END:
            phase_tag = "Phase 1"
        elif epoch <= CFG.PHASE2_END:
            phase_tag = "Phase 2"
        else:
            phase_tag = "Phase 3"

        if (ds_train is not None
                and epoch > CFG.PHASE1_END
                and epoch <= CFG.PHASE2_END
                and ((epoch == CFG.PHASE1_END + 1) or (epoch % 5 == 0))):
            frag_sampler   = compute_fragility_sampler(
                model, ds_train, y_train, device)
            current_loader = DataLoader(ds_train,
                                        sampler=frag_sampler, **kw_loader)
            print(f"  [Fragility] Sampler refreshed at epoch {epoch}")

        tr_loss, grad_stats   = train_one_epoch_repaired(
            model, current_loader, optimizer, criterion, scaler, device)
        val_loss, val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        epoch_grad_history.append(grad_stats)

        if val_metrics["f1"] > best_f1:
            best_f1      = val_metrics["f1"]
            patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
            flag = " ✓"
        else:
            patience_ctr += 1
            flag = f" [{patience_ctr}/{CFG.EARLY_STOP_PAT}]"

        elapsed = time.time() - t0
        print(f"  {epoch:4d}  {tr_loss:8.4f}  {val_loss:8.4f}  "
              f"{val_metrics['f1']:7.4f}  "
              f"{grad_stats['grad_img']:9.4f}  "
              f"{grad_stats['grad_tab']:9.4f}  "
              f"{grad_stats['grad_ratio']:7.2f}  "
              f"{phase_tag:>7}  [{elapsed:.0f}s]{flag}")

        if patience_ctr >= CFG.EARLY_STOP_PAT:
            print(f"\n  [Early Stop] Stopping at epoch {epoch}.")
            break

    print(f"\n  Loading best checkpoint (Val F1 = {best_f1:.4f}) …")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    class_names = list(le_target.classes_) if le_target else None
    print(f"\n  ── Test Set — Per-Class F1 (Repaired) ───────────────────")
    _, test_metrics = evaluate(model, test_loader, criterion, device,
                                class_names=class_names, verbose=True)
    print_metrics("Test (Repaired)", test_metrics)

    contrib = compute_logit_contributions(model, val_loader, device)
    print_modality_balance(contrib)

    avg_grad = {
        "grad_img"  : float(np.mean([g["grad_img"]   for g in epoch_grad_history])),
        "grad_tab"  : float(np.mean([g["grad_tab"]   for g in epoch_grad_history])),
        "grad_ratio": float(np.mean([g["grad_ratio"] for g in epoch_grad_history])),
    }
    return test_metrics, avg_grad, contrib

In [16]:
@torch.no_grad()
def evaluate_with_ablation(model, loader, device,
                            zero_img=False, zero_tab=False):
    model.eval()
    all_preds, all_labels = [], []

    for derm, clinic, tab_cat, tab_num, labels in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat      = tab_cat.to(device)
        tab_num      = tab_num.to(device)
        labels       = labels.to(device)

        if zero_img:
            derm   = torch.zeros_like(derm)
            clinic = torch.zeros_like(clinic)
        if zero_tab:
            tab_cat = torch.zeros_like(tab_cat)
            tab_num = torch.zeros_like(tab_num)

        with autocast():
            outputs   = model(derm, clinic, tab_cat, tab_num)
            out_final = outputs[4]

        all_preds.extend(out_final.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return compute_metrics(np.array(all_labels), np.array(all_preds))

In [17]:
def _fmt_table(headers, rows, title=""):
    if title:
        print(f"\n  ── {title} {'─'*(58 - len(title))}")
    if HAS_TABULATE:
        print(tabulate(rows, headers=headers, tablefmt="fancy_grid"))
    else:
        col_w = [max(len(h), max(len(str(r[i])) for r in rows))
                 for i, h in enumerate(headers)]
        fmt   = "  " + "  ".join(f"{{:<{w}}}" for w in col_w)
        print(fmt.format(*headers))
        print("  " + "  ".join("─" * w for w in col_w))
        for row in rows:
            print(fmt.format(*[str(x) for x in row]))

def print_final_report(before_metrics, before_grad, before_contrib,
                        after_metrics,  after_grad,  after_contrib,
                        ablation_results):

    print("\n\n" + "="*70)
    print("   MODALITY COLLAPSE DETECTION & REPAIR — FINAL REPORT")
    print("="*70)

    _fmt_table(
        ["Model", "Accuracy", "Precision", "Recall", "Macro F1", "Weighted F1"],
        [
            ["Vanilla Baseline",
             f"{before_metrics['accuracy']:.4f}",
             f"{before_metrics['precision']:.4f}",
             f"{before_metrics['recall']:.4f}",
             f"{before_metrics['f1']:.4f}",
             f"{before_metrics.get('weighted_f1', 0):.4f}"],
            ["Repaired",
             f"{after_metrics['accuracy']:.4f}",
             f"{after_metrics['precision']:.4f}",
             f"{after_metrics['recall']:.4f}",
             f"{after_metrics['f1']:.4f}",
             f"{after_metrics.get('weighted_f1', 0):.4f}"],
        ],
        title="1. Performance Comparison (Test Set)"
    )

    _fmt_table(
        ["Model", "Grad_img", "Grad_tab", "Ratio (img/tab)"],
        [
            ["Vanilla Baseline",
             f"{before_grad['grad_img']:.4f}",
             f"{before_grad['grad_tab']:.4f}",
             f"{before_grad['grad_ratio']:.2f}"],
            ["Repaired",
             f"{after_grad['grad_img']:.4f}",
             f"{after_grad['grad_tab']:.4f}",
             f"{after_grad['grad_ratio']:.2f}"],
        ],
        title="2. Gradient Dominance (epoch average)"
    )

    _fmt_table(
        ["Model", "Image Logit %", "Tabular Logit %", "Status"],
        [
            ["Vanilla Baseline",
             f"{before_contrib['img_pct']:.2f}%",
             f"{before_contrib['tab_pct']:.2f}%",
             "⚠ Collapsed" if before_contrib['img_pct'] > 65 else "✓ Balanced"],
            ["Repaired",
             f"{after_contrib['img_pct']:.2f}%",
             f"{after_contrib['tab_pct']:.2f}%",
             "⚠ Collapsed" if after_contrib['img_pct'] > 65 else "✓ Balanced"],
        ],
        title="3. Logit Contribution (val set, eval mode)"
    )

    abl_rows = [
        [lbl,
         f"{m['accuracy']:.4f}",
         f"{m['precision']:.4f}",
         f"{m['recall']:.4f}",
         f"{m['f1']:.4f}"]
        for lbl, m in ablation_results.items()
    ]
    _fmt_table(
        ["Ablation", "Accuracy", "Precision", "Recall", "Macro F1"],
        abl_rows,
        title="4. Ablation — Remove One Modality at Inference (Repaired Only)"
    )

    if "f1_per_class" in after_metrics:
        print("\n  ── 5. Per-Class F1 (Repaired, Test Set) ───────────────")
        for i, f1 in enumerate(after_metrics["f1_per_class"]):
            bar = "█" * int(f1 * 25)
            print(f"    Class {i:2d}  F1 = {f1:.4f}  {bar}")

    print("\n  ── 6. Collapse Diagnosis ──────────────────────────────────────")
    gr_b, ip_b = before_grad['grad_ratio'], before_contrib['img_pct']
    gr_a, ip_a = after_grad['grad_ratio'],  after_contrib['img_pct']
    collapse   = (gr_b > 5.0) or (ip_b > 65.0)
    print(f"    Gradient ratio (Baseline) : {gr_b:.2f}  "
          f"{'⚠ Possible collapse (>5)' if gr_b > 5 else '✓ Balanced'}")
    print(f"    Image logit % (Baseline)  : {ip_b:.2f}%  "
          f"{'⚠ Image-dominant (>65%)' if ip_b > 65 else '✓ Balanced'}")
    print(f"    Collapse detected         : "
          f"{'YES — repair was warranted.' if collapse else 'NO — naturally balanced.'}")
    print(f"\n    F1 Improvement            : {after_metrics['f1'] - before_metrics['f1']:+.4f}")
    print(f"    Gradient ratio (Repaired) : {gr_a:.2f}  "
          f"{'↓ Reduced dominance ✓' if gr_a < gr_b else '─'}")
    print(f"    Image logit %  (Repaired) : {ip_a:.2f}%")
    print(f"    Tabular logit %(Repaired) : {after_contrib['tab_pct']:.2f}%")

    print("\n  ── 7. Architecture Comparison ─────────────────────────────────")
    print(f"  {'Component':<30}  {'Baseline':^20}  {'Repaired':^22}")
    print(f"  {'─'*76}")
    rows = [
        ("Tabular encoding",
         "raw int codes",
         "FTTransformerEncoder"),
        ("Tabular backbone",
         "3-layer ReLU MLP",
         f"Transformer ×{CFG.FT_LAYERS} + mean-pool"),
        ("Fusion",
         "scalar w_img+w_tab",
         f"CrossModalAttn ({CFG.ATTN_HEADS}h, D={CFG.D_MODEL})"),
        ("CLIP alignment",
         "✗ none",
         "✓ Triple CLIP (img↔tab, img↔fused, tab↔fused)"),
        ("Gradient balance",
         "✗ none",
         "✓ Two-sided (w_img + w_tab)"),
        ("Fragility sampler",
         "✗ none",
         f"✓ α={CFG.FRAG_ALPHA}"),
        ("Modality dropout",
         "✗ none",
         f"✓ p={CFG.DROP_PROB}  (logits only)"),
        ("Phase schedule",
         "✗ flat LR",
         f"✓ 3-phase ({CFG.PHASE1_END}/{CFG.PHASE2_END})"),
        ("Backbone freeze",
         "✗ always on",
         "✓ Phase 1 frozen"),
        ("Attention module LR",
         "─",
         f"✓ LR_ATTENTION={CFG.LR_ATTENTION}"),
        ("Ablation tests",
         "✗ not run",
         "✓ image + tabular zeroing"),
    ]
    for name, base_v, rep_v in rows:
        print(f"  {name:<30}  {base_v:^20}  {rep_v:^22}")
    print("="*70)

In [18]:
def main():
    print(f"\n  Device : {CFG.DEVICE}")
    if CFG.DEVICE.type == "cuda":
        print(f"  GPU    : {torch.cuda.get_device_name(0)}")

    (train_df, valid_df, test_df,
     X_tr_cat, X_tr_num, y_train,
     X_val_cat, X_val_num, y_valid,
     X_test_cat, X_test_num, y_test,
     le_target, tab_prep) = prepare_data()

    num_classes      = len(le_target.classes_)
    cat_dims         = tab_prep.cat_dims
    baseline_tab_dim = len(CFG.CAT_COLS) + len(CFG.NUM_COLS)

    tf_train = get_transforms("train")
    tf_eval  = get_transforms("eval")

    ds_train = Derm7ptDataset(train_df, X_tr_cat,   X_tr_num,   y_train, tf_train)
    ds_valid = Derm7ptDataset(valid_df, X_val_cat,  X_val_num,  y_valid, tf_eval)
    ds_test  = Derm7ptDataset(test_df,  X_test_cat, X_test_num, y_test,  tf_eval)

    kw_eval = dict(batch_size=CFG.BATCH_SIZE,
                   num_workers=CFG.NUM_WORKERS,
                   pin_memory=True)

    val_loader  = DataLoader(ds_valid, shuffle=False, **kw_eval)
    test_loader = DataLoader(ds_test,  shuffle=False, **kw_eval)

    print("\n" + "█"*70)
    print("  STEP 1 — Vanilla Baseline  (no CLIP · no fragility · flat LR)")
    print("█"*70)

    baseline_sampler      = make_class_sampler(y_train)
    train_loader_baseline = DataLoader(ds_train,
                                        sampler=baseline_sampler, **kw_eval)
    model_baseline        = BaselineFusionModel(
        tab_dim=baseline_tab_dim, num_classes=num_classes)
    before_metrics, before_grad, before_contrib = train_model_baseline(
        model_baseline, train_loader_baseline,
        val_loader, test_loader, le_target)

    print("\n  ── Collapse Snapshot (Baseline) ─────────────────────────────")
    print(f"    Avg Gradient Ratio (img/tab) : {before_grad['grad_ratio']:.2f}")
    print(f"    Image Logit Contribution     : {before_contrib['img_pct']:.2f}%")
    print(f"    Tabular Logit Contribution   : {before_contrib['tab_pct']:.2f}%")
    print(f"    Test Macro-F1                : {before_metrics['f1']:.4f}")

    print("\n" + "█"*70)
    print("  STEP 2 — Repaired Model ")
    print("  (FT-Transformer · CrossModalAttn · TripleCLIP · 2-sided balance · 3-Phase)")
    print("█"*70)

    repaired_sampler      = make_class_sampler(y_train)
    train_loader_repaired = DataLoader(ds_train,
                                        sampler=repaired_sampler, **kw_eval)
    model_repaired        = RepairedFusionModel(
        cat_dims=cat_dims, num_classes=num_classes)
    after_metrics, after_grad, after_contrib = train_model_repaired(
        model             = model_repaired,
        base_train_loader = train_loader_repaired,
        val_loader        = val_loader,
        test_loader       = test_loader,
        ds_train          = ds_train,
        y_train           = y_train,
        le_target         = le_target,
    )

    print("\n" + "█"*70)
    print("  STEP 3 — Ablation Tests  (Repaired only)")
    print("█"*70)

    model_repaired = model_repaired.to(CFG.DEVICE)
    ablation_results = {
        "Full Model  (both modalities)":
            evaluate_with_ablation(model_repaired, test_loader, CFG.DEVICE),
        "Image Only  (tabular zeroed)":
            evaluate_with_ablation(model_repaired, test_loader, CFG.DEVICE,
                                   zero_tab=True),
        "Tabular Only (image zeroed)":
            evaluate_with_ablation(model_repaired, test_loader, CFG.DEVICE,
                                   zero_img=True),
    }
    for name, m in ablation_results.items():
        print(f"  {name:40s}  Macro-F1 = {m['f1']:.4f}")

    print_final_report(
        before_metrics, before_grad, before_contrib,
        after_metrics,  after_grad,  after_contrib,
        ablation_results,
    )

if __name__ == "__main__":
    main()


  Device : cuda
  GPU    : Tesla T4
▶  Loading metadata …
   Classes (5): ['BCC', 'MEL', 'MISC', 'NEV', 'SK']
   Train: 413 | Val: 203 | Test: 395
   Cat dims: [8, 2, 3, 3, 3, 3, 3, 4, 4, 3, 9]

██████████████████████████████████████████████████████████████████████
  STEP 1 — Vanilla Baseline  (no CLIP · no fragility · flat LR)
██████████████████████████████████████████████████████████████████████
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 177MB/s]  



  Training ► VANILLA BASELINE  (no CLIP · no fragility · flat LR)
    Ep    TrLoss   ValLoss       F1    GradImg    GradTab    Ratio
  ──────────────────────────────────────────────────────────────
     1    1.1942    1.1867   0.4051    21.8586     1.3751    15.90  [22s] ✓
     2    1.0033    1.2456   0.3745    18.9008     1.2734    14.84  [17s] [1/7]
     3    0.7369    1.1320   0.4756    14.1089     1.0893    12.95  [17s] ✓
     4    0.6375    1.1011   0.4952    12.9108     1.0565    12.22  [17s] ✓
     5    0.6064    1.1278   0.4527    12.5840     1.0021    12.56  [17s] [1/7]
     6    0.5592    1.0507   0.5886    10.0886     1.0033    10.06  [17s] ✓
     7    0.4752    1.2576   0.4810     9.4501     0.9702     9.74  [17s] [1/7]
     8    0.5368    1.0993   0.5106     9.7605     0.9957     9.80  [17s] [2/7]
     9    0.4500    1.2258   0.5192     8.3838     0.9472     8.85  [17s] [3/7]
    10    0.4081    1.0056   0.5159     8.2734     0.9213     8.98  [17s] [4/7]
    11    0.3538 

## Step 14 — Curriculum Config & Perturbation Applicators

Define `CURR`, the curriculum-specific hyperparameters (Phase 3 epochs, in-distribution and OOD perturbation ranges, reliability loss weight). Then implement the per-sample image (Gaussian noise, blur, random occlusion) and tabular (feature masking with dataset-mode defaults) perturbation functions.

In [19]:
import os, time, math, copy
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

_required_names = [
    "CFG", "AsymmetricLoss", "Derm7ptDataset", "get_transforms",
    "TabularPreprocessor", "prepare_data", "DiagnosisMultimodalNet",
    "FTTransformerEncoder", "CrossModalAttentionFusion", "TripleCLIPHead",
    "RepairedFusionModel", "compute_metrics", "print_metrics",
    "set_backbones_trainable", "compute_branch_grad_norms",
    "evaluate",
]
_missing = [n for n in _required_names if n not in globals()]
assert not _missing, (
    f"Missing  components from notebook scope: {_missing}. "
    f"Run the cell (Cell 0) before this one."
)


class CURR:
    
    PHASE3_EPOCHS   = 15         
    EARLY_STOP_PAT  = 7          

    LR_IMAGE        = CFG.LR_IMAGE
    LR_TAB          = CFG.LR_ATTENTION
    LR_FUSION       = CFG.LR_ATTENTION
    LR_CLIP         = CFG.LR_FUSION
    LR_RELIABILITY  = 5e-4       

    PERT_PROB       = 0.5        

    ID_NOISE_SIGMA  = (0.05, 0.20)
    ID_BLUR_SIGMA   = (1.0, 3.0)
    ID_OCCLUSION    = (0.10, 0.20)    
    ID_TAB_MASK     = (0.10, 0.40)    

    OOD_NOISE_SIGMA = 0.40
    OOD_BLUR_SIGMA  = 7.0
    OOD_OCCLUSION   = 0.30
    OOD_TAB_MASK    = 1.00            
    OOD_TAB_SHUFFLE = True            

    LAMBDA_REL      = 0.30

    CKPT_PATH       = CFG.CKPT_DIR / "reliability_gated_best.pt"

def _apply_image_perturbation(x, rng):
    choice = rng.choice(["noise", "blur", "occlusion"])
    if choice == "noise":
        sigma = float(rng.uniform(*CURR.ID_NOISE_SIGMA))
        return x + torch.randn_like(x) * sigma
    if choice == "blur":
        sigma = float(rng.uniform(*CURR.ID_BLUR_SIGMA))
        return _gaussian_blur_single(x, sigma)
    frac = float(rng.uniform(*CURR.ID_OCCLUSION))
    return _random_occlusion_single(x, frac, rng)


def _apply_tab_perturbation(cat, num, rng, cat_fill, num_mean):
    frac = float(rng.uniform(*CURR.ID_TAB_MASK))
    n_cat = cat.shape[0]
    n_mask_cat = int(round(frac * n_cat))
    if n_mask_cat > 0:
        idx = rng.choice(n_cat, size=n_mask_cat, replace=False)
        cat = cat.clone()
        for i in idx:
            cat[i] = int(cat_fill[i])
    n_num = num.shape[0]
    n_mask_num = int(round(frac * n_num))
    if n_mask_num > 0:
        idx = rng.choice(n_num, size=n_mask_num, replace=False)
        num = num.clone()
        for i in idx:
            num[i] = float(num_mean[i])
    return cat, num


def _gaussian_blur_single(x, sigma):
    k = max(3, int(2 * round(sigma) + 1))
    sigma = max(sigma, 0.1)
    coords = torch.arange(k, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = (g[None, None, :, None] * g[None, None, None, :]).to(x.device)
    c = x.size(0)
    w = kernel.expand(c, 1, k, k)
    return F.conv2d(x.unsqueeze(0), w, padding=k // 2, groups=c).squeeze(0)


def _random_occlusion_single(x, frac, rng):
    C, H, W = x.shape
    ph = max(1, int(H * frac))
    y0 = int(rng.integers(0, H - ph + 1))
    x0 = int(rng.integers(0, W - ph + 1))
    out = x.clone()
    out[:, y0:y0+ph, x0:x0+ph] = 0.0
    return out

## Step 15 — Phase-3 Perturbed Dataset & Collation

Wrap the clean training dataset so each `__getitem__` call randomly perturbs image or tabular features (with probability `CURR.PERT_PROB`) and returns a 6-tuple that includes a `pert_label` ∈ {0: clean, 1: image perturbed, 2: tab perturbed}. A custom collate function and per-worker RNG re-seeding ensure fresh perturbations each epoch.

In [20]:
class PerturbedPhase3Dataset(Dataset):

    def __init__(self, base_dataset: Dataset, cat_fill_values: np.ndarray,
                 num_mean_values: np.ndarray, seed: int = 42):
        self.base       = base_dataset
        self.cat_fill   = cat_fill_values     
        self.num_mean   = num_mean_values     
        self._rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        derm, clinic, cat, num, label = self.base[idx]

        r = float(self._rng.random())
        if r >= CURR.PERT_PROB:
            return derm, clinic, cat, num, label, 0

        which = int(self._rng.integers(0, 2))   
        if which == 0:
            derm   = _apply_image_perturbation(derm,   self._rng)
            clinic = _apply_image_perturbation(clinic, self._rng)
            return derm, clinic, cat, num, label, 1
        else:
            cat, num = _apply_tab_perturbation(
                cat, num, self._rng, self.cat_fill, self.num_mean)
            return derm, clinic, cat, num, label, 2


def _worker_init_fn(worker_id):
    info = torch.utils.data.get_worker_info()
    if info is not None and hasattr(info.dataset, "_rng"):
        seed = (info.seed + worker_id) % (2**32)
        info.dataset._rng = np.random.default_rng(seed)


def collate_with_pert_label(batch):
    derm   = torch.stack([b[0] for b in batch])
    clinic = torch.stack([b[1] for b in batch])
    cat    = torch.stack([b[2] for b in batch])
    num    = torch.stack([b[3] for b in batch])
    label  = torch.stack([b[4] for b in batch])
    pert   = torch.tensor([b[5] for b in batch], dtype=torch.long)
    return derm, clinic, cat, num, label, pert

## Step 16 — Reliability Head & Reliability-Gated Model

The `ReliabilityHead` is a small MLP that ingests `[img_feat, tab_feat]` and produces a 3-class soft prediction (clean / image-perturbed / tab-perturbed). Its outputs are converted into per-sample weights `w_img` and `w_tab` that multiplicatively gate the features before cross-modal attention, teaching the model to down-weight corrupted modalities at inference time.

In [21]:
class ReliabilityHead(nn.Module):

    def __init__(self, img_dim: int, tab_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim + tab_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 3),   # 3-class: clean / img_pert / tab_pert
        )

    def forward(self, img_feat, tab_feat):
        x = torch.cat([img_feat, tab_feat], dim=1)
        logits = self.net(x)                       # [B, 3]
        p = F.softmax(logits, dim=1)               # [B, 3]
        # p[:, 0] = P(clean), p[:, 1] = P(img_pert), p[:, 2] = P(tab_pert)
        w_img = (p[:, 0] + p[:, 2]).unsqueeze(1)   # trust image unless img_pert
        w_tab = (p[:, 0] + p[:, 1]).unsqueeze(1)   # trust tab unless tab_pert
        return logits, w_img, w_tab

class ReliabilityGatedModel(nn.Module):

    def __init__(self, cat_dims, num_classes):
        super().__init__()
        self.num_classes = num_classes

        self.net        = DiagnosisMultimodalNet(num_classes)
        self.ft_encoder = FTTransformerEncoder(cat_dims, num_classes)
        tab_dim         = self.ft_encoder.out_dim

        self.fusion = CrossModalAttentionFusion(
            img_dim=CFG.R_DIM, tab_dim=tab_dim, num_classes=num_classes)
        self.clip_head = TripleCLIPHead(
            img_dim=CFG.R_DIM, tab_dim=tab_dim, fused_dim=self.fusion.out_dim)

        self.reliability = ReliabilityHead(img_dim=CFG.R_DIM, tab_dim=tab_dim)

    def forward(self, derm, clinic, tab_cat, tab_num):
        B = derm.size(0)

        out_d, out_c, out_dc_raw, _, _, img_feat = self.net(derm, clinic)
        out_tab_raw, tab_feat = self.ft_encoder(tab_cat, tab_num)

        pert_logits, w_img, w_tab = self.reliability(img_feat, tab_feat)

        img_feat_gated = img_feat * w_img
        tab_feat_gated = tab_feat * w_tab

        if self.training:
            p        = CFG.DROP_PROB
            r        = torch.rand(B, device=derm.device)
            drop_img = r < p
            drop_tab = (r >= p) & (r < 2 * p)
        else:
            drop_img = torch.zeros(B, dtype=torch.bool, device=derm.device)
            drop_tab = torch.zeros(B, dtype=torch.bool, device=derm.device)

        out_dc  = out_dc_raw.clone();  out_dc[drop_img]  = 0.0
        out_tab = out_tab_raw.clone(); out_tab[drop_tab] = 0.0

        final_logits, fused_feat = self.fusion(img_feat_gated, tab_feat_gated)

        clip_loss = self.clip_head(img_feat, tab_feat, fused_feat)

        extras = dict(w_img=w_img, w_tab=w_tab, pert_logits=pert_logits)
        return out_d, out_c, out_dc, out_tab, final_logits, clip_loss, extras

In [26]:
class CompatAdapter(nn.Module):
    
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, derm, clinic, tab_cat, tab_num):
        out_d, out_c, out_dc, out_tab, final_logits, clip_loss, _extras = \
            self.model(derm, clinic, tab_cat, tab_num)
        return out_d, out_c, out_dc, out_tab, final_logits, clip_loss

    def eval(self):
        self.model.eval(); return super().eval()
    def train(self, mode=True):
        self.model.train(mode); return super().train(mode)

## Step 17 — Curriculum Training (Phases 1–3)

Implement the Phase-3 training step that adds the reliability auxiliary loss (3-class CE on `pert_label`) and tracks how often the two-sided gradient balancer fires. Then the `main_curriculum` function optionally warm-starts from a  checkpoint, runs Phases 1–2 on clean data (reliability aux loss OFF), and runs Phase 3 on the perturbed dataset with early stopping on clean val F1.

In [27]:
def train_one_epoch_phase3(model, loader, optimizer, criterion,
                             scaler, device, prev_ratio_ref):
    model.train()
    total_loss = 0.0
    grad_img_list, grad_tab_list = [], []
    balancer_fired = 0
    n_batches = 0

    rel_ce = nn.CrossEntropyLoss()

    for derm, clinic, tab_cat, tab_num, labels, pert in loader:
        derm, clinic = derm.to(device), clinic.to(device)
        tab_cat, tab_num = tab_cat.to(device), tab_num.to(device)
        labels = labels.to(device)
        pert   = pert.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out_d, out_c, out_dc, out_tab, final_logits, clip_loss, extras = \
                model(derm, clinic, tab_cat, tab_num)

            loss_img = (criterion(out_d,  labels)
                        + criterion(out_c,  labels)
                        + criterion(out_dc, labels)) / 3.0   
            loss_tab    = criterion(out_tab,      labels)
            loss_fusion = criterion(final_logits, labels)

            pr = prev_ratio_ref[0]
            if pr > 3.0:
                w_img_b = min(1.0, 3.0 / pr); balancer_fired += 1
            else:
                w_img_b = 1.0
            if pr < (1.0 / 3.0):
                w_tab_b = min(1.0, pr / 3.0); balancer_fired += 1
            else:
                w_tab_b = 1.0

            loss_rel = rel_ce(extras["pert_logits"], pert)

            loss = (w_img_b * loss_img
                    + w_tab_b * loss_tab
                    +           loss_fusion
                    + CFG.LAMBDA_CLIP  * clip_loss
                    + CURR.LAMBDA_REL  * loss_rel)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        gn = compute_branch_grad_norms(model)
        prev_ratio_ref[0] = gn["grad_ratio"]
        grad_img_list.append(gn["grad_img"])
        grad_tab_list.append(gn["grad_tab"])

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches  += 1

    return total_loss / max(1, n_batches), {
        "grad_img"       : float(np.mean(grad_img_list)),
        "grad_tab"       : float(np.mean(grad_tab_list)),
        "grad_ratio"     : float(np.mean(grad_img_list) / max(np.mean(grad_tab_list), 1e-6)),
        "balancer_fired" : balancer_fired,
        "balancer_rate"  : balancer_fired / max(1, n_batches),
    }

## Step 18 — Inference Battery & Final Evaluation

Build a battery of 14 perturbation conditions (clean, 6 in-distribution, 7 OOD) and evaluate the trained model under each. Also compute per-sample `w_img` / `w_tab` statistics to verify that the reliability gate tracks perturbation severity. Prints an F1 summary table, per-sample weight statistics, and robustness contrasts.

In [28]:
def _batch_gaussian_noise(x, sigma):
    return x + torch.randn_like(x) * sigma

def _batch_gaussian_blur(x, sigma):
    k = max(3, int(2 * round(sigma) + 1))
    sigma = max(sigma, 0.1)
    coords = torch.arange(k, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2)); g = g / g.sum()
    kernel = (g[None, None, :, None] * g[None, None, None, :]).to(x.device)
    c = x.size(1)
    w = kernel.expand(c, 1, k, k)
    return F.conv2d(x, w, padding=k // 2, groups=c)

def _batch_occlusion(x, frac):
    B, C, H, W = x.shape
    ph = max(1, int(H * frac))
    out = x.clone()
    for b in range(B):
        y0 = int(torch.randint(0, H - ph + 1, (1,)).item())
        x0 = int(torch.randint(0, W - ph + 1, (1,)).item())
        out[b, :, y0:y0+ph, x0:x0+ph] = 0.0
    return out

def _batch_tab_mask(cat, num, frac, cat_fill, num_mean, device):
    B, n_cat = cat.shape
    _,  n_num = num.shape
    cat_out = cat.clone()
    num_out = num.clone()
    if frac >= 1.0:
        for i in range(n_cat): cat_out[:, i] = int(cat_fill[i])
        num_mean_t = torch.tensor(num_mean, device=device, dtype=num.dtype)
        num_out[:] = num_mean_t.unsqueeze(0).expand(B, -1)
    else:
        mask_cat = torch.rand(B, n_cat, device=device) < frac
        mask_num = torch.rand(B, n_num, device=device) < frac
        for i in range(n_cat):
            col_mask = mask_cat[:, i]
            if col_mask.any():
                cat_out[col_mask, i] = int(cat_fill[i])
        for i in range(n_num):
            col_mask = mask_num[:, i]
            if col_mask.any():
                num_out[col_mask, i] = float(num_mean[i])
    return cat_out, num_out

def _build_eval_battery(cat_fill, num_mean, device):
    def _id_derm_noise(d, c, ca, nu):   return _batch_gaussian_noise(d, 0.15), c, ca, nu
    def _id_derm_blur(d, c, ca, nu):    return _batch_gaussian_blur(d, 2.0), c, ca, nu
    def _id_derm_occ(d, c, ca, nu):     return _batch_occlusion(d, 0.15), c, ca, nu
    def _id_clinic_noise(d, c, ca, nu): return d, _batch_gaussian_noise(c, 0.15), ca, nu
    def _id_clinic_blur(d, c, ca, nu):  return d, _batch_gaussian_blur(c, 2.0), ca, nu
    def _id_tab_mask(d, c, ca, nu):
        ca2, nu2 = _batch_tab_mask(ca, nu, 0.30, cat_fill, num_mean, device)
        return d, c, ca2, nu2

    def _ood_derm_noise(d, c, ca, nu):   return _batch_gaussian_noise(d, CURR.OOD_NOISE_SIGMA), c, ca, nu
    def _ood_derm_blur(d, c, ca, nu):    return _batch_gaussian_blur(d, CURR.OOD_BLUR_SIGMA), c, ca, nu
    def _ood_derm_occ(d, c, ca, nu):     return _batch_occlusion(d, CURR.OOD_OCCLUSION), c, ca, nu
    def _ood_clinic_noise(d, c, ca, nu): return d, _batch_gaussian_noise(c, CURR.OOD_NOISE_SIGMA), ca, nu
    def _ood_clinic_blur(d, c, ca, nu):  return d, _batch_gaussian_blur(c, CURR.OOD_BLUR_SIGMA), ca, nu
    def _ood_tab_full(d, c, ca, nu):
        ca2, nu2 = _batch_tab_mask(ca, nu, CURR.OOD_TAB_MASK, cat_fill, num_mean, device)
        return d, c, ca2, nu2
    def _ood_tab_shuffle(d, c, ca, nu):
        idx = torch.randperm(ca.size(0), device=device)
        return d, c, ca[idx], nu[idx]

    return [
        ("clean",                "none",      lambda d,c,ca,nu: (d,c,ca,nu)),
        ("ID: derm_noise_0.15",  "image",     _id_derm_noise),
        ("ID: derm_blur_2",      "image",     _id_derm_blur),
        ("ID: derm_occ_0.15",    "image",     _id_derm_occ),
        ("ID: clinic_noise_0.15","image",     _id_clinic_noise),
        ("ID: clinic_blur_2",    "image",     _id_clinic_blur),
        ("ID: tab_mask_30",      "tabular",   _id_tab_mask),
        (f"OOD: derm_noise_{CURR.OOD_NOISE_SIGMA}",   "image",   _ood_derm_noise),
        (f"OOD: derm_blur_{CURR.OOD_BLUR_SIGMA}",     "image",   _ood_derm_blur),
        (f"OOD: derm_occ_{CURR.OOD_OCCLUSION}",       "image",   _ood_derm_occ),
        (f"OOD: clinic_noise_{CURR.OOD_NOISE_SIGMA}", "image",   _ood_clinic_noise),
        (f"OOD: clinic_blur_{CURR.OOD_BLUR_SIGMA}",   "image",   _ood_clinic_blur),
        ("OOD: tab_mask_100",                         "tabular", _ood_tab_full),
        ("OOD: tab_shuffle",                          "tabular", _ood_tab_shuffle),
    ]


@torch.no_grad()
def evaluate_battery(model, loader, battery, device):
    model.eval()
    results = []
    for name, target, fn in battery:
        all_preds, all_labels = [], []
        all_w_img, all_w_tab  = [], []
        for batch in loader:
            if len(batch) == 6:
                derm, clinic, cat, num, labels, _pert = batch
            else:
                derm, clinic, cat, num, labels = batch
            derm, clinic = derm.to(device), clinic.to(device)
            cat, num = cat.to(device), num.to(device)
            labels = labels.to(device)

            derm, clinic, cat, num = fn(derm, clinic, cat, num)

            with autocast():
                out = model(derm, clinic, cat, num)
            if len(out) == 7:
                _, _, _, _, final_logits, _, extras = out
                all_w_img.append(extras["w_img"].float().cpu().numpy())
                all_w_tab.append(extras["w_tab"].float().cpu().numpy())
            else:
                final_logits = out[4]

            all_preds.extend(final_logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        m = compute_metrics(np.array(all_labels), np.array(all_preds))
        row = dict(name=name, target=target, f1=m["f1"],
                    acc=m["accuracy"], prec=m["precision"], rec=m["recall"])
        if all_w_img:
            wi = np.concatenate(all_w_img).flatten()
            wt = np.concatenate(all_w_tab).flatten()
            row.update(dict(
                w_img_mean=float(wi.mean()), w_img_std=float(wi.std()),
                w_img_min=float(wi.min()),   w_img_max=float(wi.max()),
                w_tab_mean=float(wt.mean()), w_tab_std=float(wt.std()),
                w_tab_min=float(wt.min()),   w_tab_max=float(wt.max()),
            ))
        results.append(row)
    return results

In [29]:
def main_curriculum():
    device = CFG.DEVICE
    print(f"\n  Device : {device}")
    if device.type == "cuda":
        print(f"  GPU    : {torch.cuda.get_device_name(0)}")

    (train_df, valid_df, test_df,
     X_tr_cat, X_tr_num, y_train,
     X_val_cat, X_val_num, y_valid,
     X_test_cat, X_test_num, y_test,
     le_target, tab_prep) = prepare_data()

    num_classes = len(le_target.classes_)
    cat_dims    = tab_prep.cat_dims

    cat_fill = np.array([
        tab_prep.label_encoders[col].transform([tab_prep.fill_values[col]])[0]
        for col in CFG.CAT_COLS
    ], dtype=np.int64)
    num_mean = np.zeros(len(CFG.NUM_COLS), dtype=np.float32)  # scaler was fit on train

    tf_train = get_transforms("train")
    tf_eval  = get_transforms("eval")
    ds_train = Derm7ptDataset(train_df, X_tr_cat,  X_tr_num,  y_train, tf_train)
    ds_valid = Derm7ptDataset(valid_df, X_val_cat, X_val_num, y_valid, tf_eval)
    ds_test  = Derm7ptDataset(test_df,  X_test_cat, X_test_num, y_test, tf_eval)

    kw_eval = dict(batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS,
                   pin_memory=True)
    val_loader  = DataLoader(ds_valid, shuffle=False, **kw_eval)
    test_loader = DataLoader(ds_test,  shuffle=False, **kw_eval)

    print("\n" + "█"*70)
    print("  Reliability-Gated — Curriculum Training")
    print("█"*70)

    model = ReliabilityGatedModel(cat_dims=cat_dims, num_classes=num_classes).to(device)

    v5_ckpt = CFG.CKPT_DIR / "repaired_fusion__best.pt"
    has_v5  = v5_ckpt.exists()
    if has_v5:
        print(f"  Found v5 checkpoint at {v5_ckpt}")
        print(f"  Warm-starting components (net, ft_encoder, fusion, clip_head)")
        v5_state = torch.load(v5_ckpt, map_location=device)
        our_state = model.state_dict()
        matched = 0
        for k, v in v5_state.items():
            if k in our_state and our_state[k].shape == v.shape:
                our_state[k] = v
                matched += 1
        model.load_state_dict(our_state)
        print(f"  Matched {matched} tensors from checkpoint.\n")
        skip_p12 = True
    else:
        print("  No checkpoint found — will train Phases 1 & 2 from scratch.\n")
        skip_p12 = False

    criterion = AsymmetricLoss(gamma_neg=2.0, gamma_pos=1.0, eps=0.0)

    optimizer = AdamW([
        {"params": model.net.parameters(),         "lr": CURR.LR_IMAGE},
        {"params": model.ft_encoder.parameters(),  "lr": CURR.LR_TAB},
        {"params": model.fusion.parameters(),      "lr": CURR.LR_FUSION},
        {"params": model.clip_head.parameters(),   "lr": CURR.LR_CLIP},
        {"params": model.reliability.parameters(), "lr": CURR.LR_RELIABILITY},
    ], weight_decay=CFG.WEIGHT_DECAY)

    scaler = GradScaler()

    from torch.utils.data import WeightedRandomSampler
    class_counts = np.bincount(y_train, minlength=num_classes)
    class_w      = 1.0 / np.maximum(class_counts, 1)
    sample_w     = class_w[y_train]
    clean_sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.float64),
        num_samples=len(sample_w), replacement=True)
    clean_loader = DataLoader(ds_train, sampler=clean_sampler, **kw_eval)

    prev_ratio_ref = [1.0]
    if not skip_p12:
        print("─"*70)
        print("  PHASE 1 & 2 — clean-data training (using recipe)")
        print("─"*70)
        for epoch in range(1, CFG.PHASE2_END + 1):
            t0 = time.time()
            if epoch == 1:
                set_backbones_trainable(model, False)
                print("  [Phase 1] Backbones frozen.")
            elif epoch == CFG.PHASE1_END + 1:
                set_backbones_trainable(model, True)
                print("  [Phase 2] Backbones unfrozen.")

            model.train()
            total_loss, n = 0.0, 0
            for derm, clinic, cat, num, labels in clean_loader:
                derm, clinic = derm.to(device), clinic.to(device)
                cat, num = cat.to(device), num.to(device)
                labels = labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    out_d, out_c, out_dc, out_tab, final_logits, clip_loss, _extras = \
                        model(derm, clinic, cat, num)
                    loss_img = (criterion(out_d,  labels)
                                + criterion(out_c,  labels)
                                + criterion(out_dc, labels)) / 3.0
                    loss_tab    = criterion(out_tab,      labels)
                    loss_fusion = criterion(final_logits, labels)
                    loss = loss_img + loss_tab + loss_fusion + CFG.LAMBDA_CLIP * clip_loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update()
                total_loss += loss.item(); n += 1

            adapter = CompatAdapter(model)
            val_loss, val_m = evaluate(adapter, val_loader, criterion, device)
            print(f"  P1-2 epoch {epoch:>3}  tr={total_loss/max(1,n):.4f}  "
                  f"val={val_loss:.4f}  F1={val_m['f1']:.4f}  [{time.time()-t0:.0f}s]")

    print("\n" + "─"*70)
    print(f"  PHASE 3 — Perturbation curriculum ({CURR.PHASE3_EPOCHS} epochs)")
    print("  - 50% of training samples perturbed on-the-fly per batch")
    print("  - Reliability aux loss ACTIVE")
    print("  - Early stop on clean val F1 (patience 7)")
    print("─"*70)

    pert_ds = PerturbedPhase3Dataset(ds_train, cat_fill, num_mean,
                                       seed=CFG.SEED)
    p3_loader = DataLoader(
        pert_ds,
        sampler     = WeightedRandomSampler(
            weights=torch.tensor(sample_w, dtype=torch.float64),
            num_samples=len(sample_w), replacement=True),
        batch_size  = CFG.BATCH_SIZE,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = True,
        collate_fn  = collate_with_pert_label,
        worker_init_fn = _worker_init_fn,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CURR.PHASE3_EPOCHS, eta_min=1e-6)

    best_val_f1  = 0.0
    patience_ctr = 0
    best_state   = None

    print(f"  {'Ep':>3}  {'TrLoss':>8}  {'ValLoss':>8}  {'CleanF1':>8}  "
          f"{'GradRatio':>9}  {'BalFire%':>9}  {'Elapsed':>8}")
    print(f"  {'─'*72}")

    for ep in range(1, CURR.PHASE3_EPOCHS + 1):
        t0 = time.time()
        tr_loss, stats = train_one_epoch_phase3(
            model, p3_loader, optimizer, criterion, scaler, device, prev_ratio_ref)
        adapter = CompatAdapter(model)
        val_loss, val_m = evaluate(adapter, val_loader, criterion, device)
        scheduler.step()

        improved = val_m["f1"] > best_val_f1
        if improved:
            best_val_f1 = val_m["f1"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, CURR.CKPT_PATH)
            patience_ctr = 0
            flag = " ✓"
        else:
            patience_ctr += 1
            flag = f" ({patience_ctr}/{CURR.EARLY_STOP_PAT})"

        print(f"  {ep:>3}  {tr_loss:>8.4f}  {val_loss:>8.4f}  "
              f"{val_m['f1']:>8.4f}  {stats['grad_ratio']:>9.2f}  "
              f"{100 * stats['balancer_rate']:>8.1f}%  {time.time()-t0:>7.1f}s{flag}")

        if patience_ctr >= CURR.EARLY_STOP_PAT:
            print(f"\n  [Early stop] No improvement for {CURR.EARLY_STOP_PAT} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\n  Phase 3 complete. Best clean val F1 = {best_val_f1:.4f}")

    print("\n" + "█"*70)
    print("  FINAL EVALUATION — clean + in-distribution + OOD")
    print("█"*70)

    battery = _build_eval_battery(cat_fill, num_mean, device)
    results = evaluate_battery(model, test_loader, battery, device)

    print(f"\n  {'Condition':<36} {'Target':<10} {'F1':>7}  {'Δclean':>8}")
    print(f"  {'-'*66}")
    clean_f1 = next(r["f1"] for r in results if r["name"] == "clean")
    for r in results:
        delta = f"{r['f1'] - clean_f1:+.4f}" if r["name"] != "clean" else "—"
        print(f"  {r['name']:<36} {r['target']:<10} {r['f1']:>7.4f}  {delta:>8}")

    print("\n  " + "─"*70)
    print("  PER-SAMPLE FUSION-WEIGHT STATISTICS")
    print("  " + "─"*70)
    print(f"  {'Condition':<36} {'Target':<10} {'w_img':>18} {'w_tab':>18}")
    print(f"  {'-'*82}")
    print(f"  {'':<36} {'':<10} {'mean±std (min,max)':>18} {'mean±std (min,max)':>18}")

    def _fmt_w(mean, std, mn, mx):
        return f"{mean:.2f}±{std:.2f} ({mn:.2f},{mx:.2f})"

    for r in results:
        if "w_img_mean" not in r:
            continue
        wi = _fmt_w(r["w_img_mean"], r["w_img_std"], r["w_img_min"], r["w_img_max"])
        wt = _fmt_w(r["w_tab_mean"], r["w_tab_std"], r["w_tab_min"], r["w_tab_max"])
        print(f"  {r['name']:<36} {r['target']:<10} {wi:>18} {wt:>18}")

    print("\n  " + "─"*70)
    print("  ROBUSTNESS CONTRASTS  (F1 drop from clean baseline)")
    print("  " + "─"*70)
    id_img  = [r for r in results if r["name"].startswith("ID:") and r["target"] == "image"]
    id_tab  = [r for r in results if r["name"].startswith("ID:") and r["target"] == "tabular"]
    ood_img = [r for r in results if r["name"].startswith("OOD:") and r["target"] == "image"]
    ood_tab = [r for r in results if r["name"].startswith("OOD:") and r["target"] == "tabular"]

    def _avg_drop(rs):
        if not rs: return 0.0
        return float(np.mean([clean_f1 - r["f1"] for r in rs]))

    print(f"  In-dist image  : avg F1 drop = {_avg_drop(id_img):+.4f}")
    print(f"  In-dist tab    : avg F1 drop = {_avg_drop(id_tab):+.4f}")
    print(f"  OOD image      : avg F1 drop = {_avg_drop(ood_img):+.4f}")
    print(f"  OOD tab        : avg F1 drop = {_avg_drop(ood_tab):+.4f}")
    print()
    print("  Interpretation:")
    print("   • Small drops = robust. OOD drops ≈ ID drops → generalised robustness.")
    print("   • w_img should fall under image perturbations; w_tab under tab perturbations.")
    print("   • Large std in w_img/w_tab across samples = the gate is genuinely per-sample.")
    print()

    return model, results

if __name__ == "__main__":
    model_rg, results_rg = main_curriculum()


  Device : cuda
  GPU    : Tesla T4
▶  Loading metadata …
   Classes (5): ['BCC', 'MEL', 'MISC', 'NEV', 'SK']
   Train: 413 | Val: 203 | Test: 395
   Cat dims: [8, 2, 3, 3, 3, 3, 3, 4, 4, 3, 9]

██████████████████████████████████████████████████████████████████████
  Reliability-Gated — Curriculum Training
██████████████████████████████████████████████████████████████████████
  No checkpoint found — will train Phases 1 & 2 from scratch.

──────────────────────────────────────────────────────────────────────
  PHASE 1 & 2 — clean-data training (using recipe)
──────────────────────────────────────────────────────────────────────
  [Phase 1] Backbones frozen.
  P1-2 epoch   1  tr=0.7385  val=1.2696  F1=0.4482  [13s]
  P1-2 epoch   2  tr=0.5963  val=1.1077  F1=0.5771  [13s]
  P1-2 epoch   3  tr=0.5083  val=1.0822  F1=0.5722  [13s]
  P1-2 epoch   4  tr=0.4526  val=1.0108  F1=0.5851  [13s]
  P1-2 epoch   5  tr=0.4129  val=1.0391  F1=0.5833  [13s]
  [Phase 2] Backbones unfrozen.
  P1-2 epoch

## Step 19 — Per-Sample Reliability Gate Demo

For one correctly-predicted test sample per class, sweep image blur (σ = 1…5) and tabular mask fraction (20–100%) and print `w_img`, `w_tab`, and the reliability head's 3-class softmax at each severity level. Expected behaviour: `w_img` should fall and `P(img⚠)` should rise with blur; `w_tab` should fall and `P(tab⚠)` should rise with masking.

In [30]:
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader

_required = [
    "CFG", "CURR", "ReliabilityGatedModel", "prepare_data",
    "Derm7ptDataset", "get_transforms",
    "_batch_gaussian_blur", "_batch_tab_mask",
]
_missing = [n for n in _required if n not in globals()]
assert not _missing, (
    f"Missing names from scope: {_missing}. "
    f"Run the cell AND the Phase-3 curriculum cell before this one."
)


class DEMO:
    N_PER_CLASS           = 1          # one sample per class
    MAX_RETRIES_PER_CLASS = 20         # cap random search for correct sample
    BLUR_SIGMAS           = [1, 2, 3, 4, 5]
    TAB_MASK_FRACS        = [0.2, 0.4, 0.6, 0.8, 1.0]
    SEED                  = CFG.SEED


@torch.no_grad()
def _model_forward(model, derm, clinic, cat, num):
    out = model(derm, clinic, cat, num)
    _, _, _, _, final_logits, _, extras = out
    probs = F.softmax(final_logits, dim=1)                # [1, C]
    conf, pred = probs.max(dim=1)
    w_img = extras["w_img"].squeeze().item()
    w_tab = extras["w_tab"].squeeze().item()
    pert_probs = F.softmax(extras["pert_logits"], dim=1).squeeze()  # [3]
    return (int(pred.item()),
            float(conf.item()),
            float(w_img),
            float(w_tab),
            [float(pert_probs[i].item()) for i in range(3)])


def _sample_to_device(ds, idx, device):
    derm, clinic, cat, num, label = ds[idx]
    return (derm.unsqueeze(0).to(device),
            clinic.unsqueeze(0).to(device),
            cat.unsqueeze(0).to(device),
            num.unsqueeze(0).to(device),
            int(label.item()))


def _clean_pred_info(model, ds, idx, device):
    derm, clinic, cat, num, label = _sample_to_device(ds, idx, device)
    pred, conf, _, _, _ = _model_forward(model, derm, clinic, cat, num)
    return pred, conf, (pred == label)


def _select_samples_per_class(model, ds, y_test, num_classes, device, seed):
    rng = np.random.default_rng(seed)
    chosen = {}

    for c in range(num_classes):
        class_idxs = np.where(y_test == c)[0]
        if len(class_idxs) == 0:
            chosen[c] = (None, False)
            continue

        rng.shuffle(class_idxs)
        picked_idx = None
        for try_i, idx in enumerate(class_idxs[:DEMO.MAX_RETRIES_PER_CLASS]):
            _, _, is_correct = _clean_pred_info(model, ds, int(idx), device)
            if is_correct:
                picked_idx = int(idx)
                break

        if picked_idx is not None:
            chosen[c] = (picked_idx, True)
        else:
            best_idx, best_conf = None, -1.0
            for idx in class_idxs:
                _, conf, _ = _clean_pred_info(model, ds, int(idx), device)
                if conf > best_conf:
                    best_conf = conf
                    best_idx = int(idx)
            chosen[c] = (best_idx, False)

    return chosen

@torch.no_grad()
def per_sample_reliability_demo():
    device = CFG.DEVICE

    (_, _, test_df,
     _, _, _,
     _, _, _,
     X_test_cat, X_test_num, y_test,
     le_target, tab_prep) = prepare_data()

    num_classes = len(le_target.classes_)
    class_names = list(le_target.classes_)
    cat_dims    = tab_prep.cat_dims

    tf_eval = get_transforms("eval")
    ds_test = Derm7ptDataset(test_df, X_test_cat, X_test_num, y_test, tf_eval)

    cat_fill = np.array([
        tab_prep.label_encoders[col].transform([tab_prep.fill_values[col]])[0]
        for col in CFG.CAT_COLS
    ], dtype=np.int64)
    num_mean = np.zeros(len(CFG.NUM_COLS), dtype=np.float32)

    model = ReliabilityGatedModel(cat_dims=cat_dims,
                                     num_classes=num_classes).to(device)
    ckpt_path = CURR.CKPT_PATH
    assert ckpt_path.exists(), (
        f"No trained checkpoint at {ckpt_path}. "
        f"Run the curriculum cell first."
    )
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    print(f"Loaded reliability-gated model from {ckpt_path}\n")

    chosen = _select_samples_per_class(model, ds_test, y_test, num_classes,
                                         device, DEMO.SEED)

    print("="*95)
    print(f"  PER-SAMPLE RELIABILITY GATE DEMO")
    print(f"  One sample per class, seeded ({DEMO.SEED}).")
    print(f"  Image perturbation: derm+clinic Gaussian blur σ ∈ {DEMO.BLUR_SIGMAS}")
    print(f"  Tab   perturbation: mask fraction ∈ {DEMO.TAB_MASK_FRACS}")
    print("="*95)

    for c in range(num_classes):
        idx, has_correct = chosen[c]
        if idx is None:
            print(f"\n[No test samples for class {class_names[c]}]\n")
            continue

        derm, clinic, cat, num, label = _sample_to_device(ds_test, idx, device)

        flag = " correct clean prediction" if has_correct else " no correct sample found — showing highest-confidence"
        print()
        print("═"*95)
        print(f"  SAMPLE #{idx}    True class: {class_names[label]}    ({flag})")
        print("═"*95)

        hdr = (f"  {'Condition':<22} "
               f"{'Pred':<7} {'Conf':>6}  "
               f"{'w_img':>6} {'w_tab':>6}   "
               f"{'P(clean)':>9} {'P(img⚠)':>9} {'P(tab⚠)':>9}")
        print(hdr)
        print(f"  {'-'*90}")

        pred, conf, w_img, w_tab, pp = _model_forward(model, derm, clinic, cat, num)
        pred_str = class_names[pred] + (" ✓" if pred == label else " ✗")
        print(f"  {'clean':<22} "
              f"{pred_str:<7} {conf:>6.3f}  "
              f"{w_img:>6.3f} {w_tab:>6.3f}   "
              f"{pp[0]:>9.3f} {pp[1]:>9.3f} {pp[2]:>9.3f}")

        for sigma in DEMO.BLUR_SIGMAS:
            derm_p   = _batch_gaussian_blur(derm,   float(sigma))
            clinic_p = _batch_gaussian_blur(clinic, float(sigma))
            pred, conf, w_img, w_tab, pp = _model_forward(
                model, derm_p, clinic_p, cat, num)
            pred_str = class_names[pred] + (" ✓" if pred == label else " ✗")
            print(f"  {'image blur σ=' + str(sigma):<22} "
                  f"{pred_str:<7} {conf:>6.3f}  "
                  f"{w_img:>6.3f} {w_tab:>6.3f}   "
                  f"{pp[0]:>9.3f} {pp[1]:>9.3f} {pp[2]:>9.3f}")

        for frac in DEMO.TAB_MASK_FRACS:
            cat_p, num_p = _batch_tab_mask(
                cat, num, float(frac), cat_fill, num_mean, device)
            pred, conf, w_img, w_tab, pp = _model_forward(
                model, derm, clinic, cat_p, num_p)
            pred_str = class_names[pred] + (" ✓" if pred == label else " ✗")
            pct = int(round(frac * 100))
            print(f"  {'tab mask ' + str(pct) + '%':<22} "
                  f"{pred_str:<7} {conf:>6.3f}  "
                  f"{w_img:>6.3f} {w_tab:>6.3f}   "
                  f"{pp[0]:>9.3f} {pp[1]:>9.3f} {pp[2]:>9.3f}")

    print("\n" + "─"*95)
    print("  INTERPRETATION")
    print("─"*95)
    print("   • w_img should DECLINE with increasing image blur σ.")
    print("   • P(img⚠) should RISE with increasing image blur σ.")
    print("   • w_tab should DECLINE with increasing tab mask fraction.")
    print("   • P(tab⚠) should RISE with increasing tab mask fraction.")
    print("   • Prediction may flip at high severity — that's OK; the gate's")
    print("     job is to reflect modality reliability, not to save every prediction.")
    print("   • Weights that DON'T move with severity = the reliability head")
    print("     hasn't learned to detect that perturbation class.")
    print()

## Step 20 — Run Full Pipeline

Execute all three stages in order: the baseline vs. repaired comparison, the curriculum training, and the per-sample reliability demo.

In [31]:
if __name__ == "__main__":
    per_sample_reliability_demo()

▶  Loading metadata …
   Classes (5): ['BCC', 'MEL', 'MISC', 'NEV', 'SK']
   Train: 413 | Val: 203 | Test: 395
   Cat dims: [8, 2, 3, 3, 3, 3, 3, 4, 4, 3, 9]
Loaded reliability-gated model from /kaggle/working/checkpoints/reliability_gated_best.pt

  PER-SAMPLE RELIABILITY GATE DEMO
  One sample per class, seeded (7).
  Image perturbation: derm+clinic Gaussian blur σ ∈ [1, 2, 3, 4, 5]
  Tab   perturbation: mask fraction ∈ [0.2, 0.4, 0.6, 0.8, 1.0]

═══════════════════════════════════════════════════════════════════════════════════════════════
  SAMPLE #3    True class: BCC    ( correct clean prediction)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Condition              Pred      Conf   w_img  w_tab    P(clean)   P(img⚠)   P(tab⚠)
  ------------------------------------------------------------------------------------------
  clean                  BCC ✓    0.893   0.993  0.561       0.554     0.007     0.439
  image blur σ=1         B